<span style="font-size:10pt">&copy; 2025 Michał Bukowski (m.bukowski@uj.edu.pl) ZBA WBBiB UJ</span>

# Uwagi dotyczące zadania
---

<font size=2>Biblioteka `PyHMMER` (https://pyhmmer.readthedocs.io) jest ścisłym odpowiednikiem narzędzi `HMMER` (http://hmmer.org) i bazuje na ich kodzie. Zanim uruchomisz jakiekolwiek narzędzie (np. `hmmsearch`) zapoznaj się z jego dokumentacją, szczególnie w odniesieniu do możliwych do użycia argumentów i ich znaczenia (http://eddylab.org/software/hmmer/Userguide.pdf, rozdział `Manual pages for HMMER programs`).
</font>

## 1. Parsowanie plików z modelami HMM
- Obiekt typu `HMMFile` [[doc](https://pyhmmer.readthedocs.io/en/stable/api/plan7/parsers.html)] jest dedykowany do pracy z plikami `*.hmm`:
```Python
# Podejście nr 1
with pyhmmer.plan7.HMMFile('ścieżka pliku HMM') as hmm_f:
    # ...

# Podejście nr 2
hmm_f = pyhmmer.plan7.HMMFile('ścieżka pliku HMM')
# ...
hmm_f.close()
```
- Obiekt typu `HMMFile` posiada metodę `read()`, która każdorazowo odczytuje i zwraca kolejny model zapisany w pliku jako obiekt typu `HMM` [[doc](https://pyhmmer.readthedocs.io/en/stable/api/plan7/hmms.html#pyhmmer.plan7.HMM)]. Ten sam efekt uzyskuje się iterując po otwartym pliku:
```Python
for hmm in hmm_f:
    name = hmm.name.decode()
    accn = hmm.accession.decode()
    desc = hmm.description.decode()
    print(f'Model: {name} ({accn}), {desc}.')
```

## 2. Parsowanie plików z sekwencjami w formacie FASTA
- Obiekt typu `SequenceFile` [[doc](https://pyhmmer.readthedocs.io/en/stable/api/easel/parsers.html#pyhmmer.easel.SequenceFile)] można wykorzystać do parsowania plików FASTA:
```Python
# Podejście nr 1
with pyhmmer.easel.SequenceFile('ścieżka pliku FASTA', digital=True) as seq_f:
    # ...

# Podejście nr 2
seq_f = pyhmmer.easel.SequenceFile('ścieżka pliku FASTA', digital=True)
# ...
seq_f.close()
```
- Obiekt typu `SequenceFile` posiada metodę `read()`, która każdorazowo odczytuje i zwraca kolejny model zapisany w pliku jako obiekt typu `Sequence` [[doc](https://pyhmmer.readthedocs.io/en/stable/api/easel/seq.html#pyhmmer.easel.Sequence)]. Dokładnie ten sam efekt uzyskuje się iterując po otwartym pliku:
```Python
for seq in seq_f:
    seqid = seq.name.decode()
    title = seq.description.decode()
    print(f'Sequence: {seqid} ({title}).')
```
- Metodą `write()` można zapisać sekwencję (w trybie binarnym) do wybranego pliku:
```Python
with open('ścieżka pliku FASTA', 'wb') as faa_f:
    seq.write(faa_f)
```

## 3. Korzystanie z narzędzi HMMER
- Funkcje takie jak `hmmsearch()` [[doc](https://pyhmmer.readthedocs.io/en/stable/api/hmmer/profile.html#pyhmmer.hmmer.hmmsearch)] czy `hmmscan()` [[doc](https://pyhmmer.readthedocs.io/en/stable/api/hmmer/profile.html#pyhmmer.hmmer.hmmscan)], będące odpowiednikiem narzędzi HMMER o dokładnie takiej samej nazwie, zwracają obiekt typu `TopHits` [[doc](https://pyhmmer.readthedocs.io/en/stable/api/plan7/results.html#pyhmmer.plan7.TopHits)], po których można dokonywać iteracji:
```Python
for hits in pyhmmer.hmmsearch(hmm, seq_f, cpus=8, E=0.01):
    # ...
```
- W każdej iteracji zwracany jest obiekty typu `Hit` [[doc](https://pyhmmer.readthedocs.io/en/stable/api/plan7/results.html#pyhmmer.plan7.Hit)], odnoszący się do jednej sekwencji aminokwasowej z bazy danych, który zawiera m.in. informacje o samej sekwencji (`name`, `description`):
```Python
    for hit in hits:
        seqid = hit.name.decode()
        title = hit.description.decode()
        print(f'Target sequence: {seqid} ({title})')
        # ...
```
- Obiekt `Hit` zawiera również informacje o dopasowanych do sekwencji domenach w obiekcie typu `Domains` [[doc](https://pyhmmer.readthedocs.io/en/stable/api/plan7/results.html#pyhmmer.plan7.Domains)], który jest również iterowalny. W procesie iteracji otrzymujemy obiekty typu `Domain` [[doc](https://pyhmmer.readthedocs.io/en/stable/api/plan7/results.html#pyhmmer.plan7.Domain)]:
```Python
        for dom in hit.domains:
            i_eval   = dom.i_evalue
            env_from = dom.env_from
            env_to   = dom.env_to
            print(f'Domain envelope coordinates in the target sequence: {env_from}-{env_to}')
            print(f'Domain hit independent E_value: {i_eval}')
```
- Każdy obiekt typu `Domain` zawiera również obiekt typu `Alignment` [[doc](https://pyhmmer.readthedocs.io/en/stable/api/plan7/results.html#pyhmmer.plan7.Alignment)], który zawiera bardziej szczegółowe informacje o dopasowaniu pomiędzy fragmentem sekwencji z bazy danych a modelem HMM domeny:
```Python
            hmm_len  = dom.alignment.hmm_length
            hmm_from = dom.alignment.hmm_from
            hmm_to   = dom.alignment.hmm_to
            
            seq_len  = dom.alignment.target_length
            seq_from = dom.alignment.target_from
            seq_to   = dom.alignment.target_to
            
            print(f'Aligned HMM length: {hmm_len}, aligned portion: {hmm_from}-{hmm_to}')
            print(f'Aligned sequence length: {seq_len}, aligned portion: {seq_from}-{seq_to}')
```
- Na podstawie powyższych współrzędnych możemy łatwo obliczyć pokrycie modelu domeny przez dopasowany fragment sekwencji aminokwasowej:
$${cov} = \frac {|to - from| + 1} {length} $$

# Zadanie
---

UWAGA: <u>wszystkie</u> poniższe kroki powinny być zaimplementowane <u>w tym notatniku</u>:
1. Odpowiednimi komendami Bash (`mkdir -p` i `wget`) utwórz katalogi `output/` i `dbs/` a do tego drugiego pobierz bazę danych Pfam (`Pfam-A.hmm`) ze strony serwisu `InterPro` [[tutaj](https://www.ebi.ac.uk/interpro)]. Kolejne kroki wykonaj z pomocą biblioteki `PyHMMER`.
3. Wyszukaj w bazie Pfam (`Pfam-A.hmm`) model o nazwie `Peptidase_M23` o numerze dostępu (accession number, accn) `PF01551` (w bazie danych będą to numery z wersją, czyli w formacie `accn.ver`).
5. Wykorzystaj ten model do przeszukania wszystkich sekwencji znajdujących się we wszystkich plikach FASTA w katalogu `prots/`. Tutaj możesz skorzystać z dobrodziejstw klasy `Path` z biblioteki `pathlib` i metody `glob()` [[doc](https://docs.python.org/3/library/pathlib.html#pathlib.Path.glob)] obiektów tegoż typu.
6. Weź pod uwagę tylko te dopasowania, które pokrywają znaczącą część modelu domeny &mdash; `cov` &ge; `90%` &mdash; i których `iE-value` &le; `0.01`.
7. Wyodrębnij sekwencje zawierające takie dopasowania do osobnego pliku FASTA `output/extracted.faa`.
8. Przeszukaj wyodrębnione sekwencje względem całej bazy Pfam. Zastosuj takie same kryteria `pokrycia` i `iE-value` jak te opisane wcześniej.
9. Wylicz <u>częstości współwystępowania</u> domeny `Peptidase_M23` z innymi domenami w przeanalizowanych sekwencjach aminokwasowych. Uwzględnij częstość występowania samej domeny `Peptidase_M23` jako jedynej domeny w sekwencjach aminokwasowych z bazy danych.
10. Korzystając z biblioteki `Matplotlib`, zobrazuj otrzymane wartości na wykresie słupkowym, którego słupki będą podpisane nazwami domen (X-tick labels) obróconymi o 45&deg;.

In [ ]:
# 1 Zadanie
!mkdir -p output dbs # stworzenie folderów
!wget -P dbs ftp://ftp.ebi.ac.uk/pub/databases/Pfam/current_release/Pfam-A.hmm.gz #pobranie bazy danych
!gunzip /dbs/Pfam-A.hmm.gz #rozpakowanie pliku


In [ ]:
import pyhmmer
from pathlib import Path

target_model_hmm = None
target_model_name = "Peptidase_M23"
target_model_accn = "PF01551"

with pyhmmer.plan7.HMMFile("dbs/Pfam-A.hmm") as hmm_f:
    for hmm in hmm_f:
        if hmm.name.decode() == target_model_name and hmm.accession.decode().startswith(target_model_accn):
            print(f"Name: {hmm.name.decode()}, Accession: {hmm.accession.decode()}")
            target_model_hmm = hmm 
            break 

prots_dir = Path("prots")
seq_files = list(prots_dir.glob("*.faa"))
all_sequences = []

for file_path in seq_files:
        with pyhmmer.easel.SequenceFile(file_path, digital=True) as seq_f:
            all_sequences.extend(seq_f)

top_hits = pyhmmer.hmmsearch(target_model_hmm, all_sequences, cpus=8, E=0.01)

hit_sequences = []
for hits in top_hits:
     for hit in hits:
        for dom in hit.domains:
            if dom.i_evalue <= 0.01:
                if ((abs(dom.alignment.hmm_to - dom.alignment.hmm_from) + 1) / dom.alignment.hmm_length) > 0.9:
                    hit_sequences.append(hit)

print(len(hit_sequences))
print(len(all_sequences))

print(type(all_sequences[0]))
# trzeba wziąć sekwencje z digital sequence z all sequences -> później wziąć sekwencje z hit_sequences i wziąc sekwencje z pliku all_sequences i zapisać do pliku, chyba (razem z nazwami i id itd)

Name: Peptidase_M23, Accession: PF01551.30
300
276721
<class 'pyhmmer.easel.DigitalSequence'>
